# **Configuração**

In [68]:
%%writefile config.py

# ==========================================
# CONFIGURAÇÕES GERAIS
# ==========================================

ANOS_SIMULACAO = 100

# ==========================================
# POPULAÇÃO INICIAL
# ==========================================

POPULACAO_INICIAL = 10000

CRIANCAS_INICIAL = 2500
ADULTOS_INICIAL = 6000
IDOSOS_INICIAL = 1500

# ==========================================
# SEGURANÇA ALIMENTAR
# ==========================================

# Inspirado na EBIA / dados brasileiros recentes

INSEGURANCA_GRAVE_INICIAL = 0.032
INSEGURANCA_MODERADA_INICIAL = 0.068
INSEGURANCA_LEVE_INICIAL = 0.20

# ==========================================
# RECURSOS INICIAIS
# ==========================================

ALIMENTOS_INICIAIS = 250000

ENERGIA_INICIAL = 100000

MINERIO_INICIAL = 100000

CAPITAL_INICIAL = 10000

# ==========================================
# EMPRESAS
# ==========================================

EMPRESAS_INICIAIS = 100

# ==========================================
# INDICADORES SOCIAIS
# ==========================================

EDUCACAO_INICIAL = 50

SAUDE_INICIAL = 48

INOVACAO_INICIAL = 50

FELICIDADE_INICIAL = 58

# ==========================================
# EVENTOS ECONÔMICOS
# ==========================================

PROB_BOOM = 0.03

PROB_RECESSAO = 0.05

# ==========================================
# SISTEMA FINANCEIRO
# ==========================================

JUROS_BASE = 5.0

CREDITO_BASE = 10000

# ==========================================
# COMÉRCIO
# ==========================================

LIMITE_EXPORTACAO_ALIMENTOS = 300000

LIMITE_IMPORTACAO_ALIMENTOS = 100000

VOLUME_COMERCIO_ALIMENTOS = 20000

VOLUME_COMERCIO_ENERGIA = 10000

VOLUME_COMERCIO_MINERIO = 10000

Overwriting config.py


### **Migração**

In [70]:
%%writefile migracao.py

# ==========================================
# MIGRAÇÃO V11 (Forçada e Direta)
# ==========================================

def atualizar_migracao(ilhas):
    # Encontra a ilha com maior felicidade e a com menor felicidade
    mais_feliz = max(ilhas, key=lambda x: x.felicidade)
    menos_feliz = min(ilhas, key=lambda x: x.felicidade)

    if mais_feliz != menos_feliz:
        # Se houver diferença de felicidade, há fluxo migratório
        diferenca = mais_feliz.felicidade - menos_feliz.felicidade

        if diferenca > 0.5:
            # 2% da população adulta da ilha menos feliz migra para a mais feliz
            migrantes = int(menos_feliz.adultos * 0.02)

            if migrantes > 0 and menos_feliz.adultos > migrantes:
                menos_feliz.adultos -= migrantes
                mais_feliz.adultos += migrantes

                menos_feliz.emigrantes += migrantes
                mais_feliz.imigrantes += migrantes

    return ilhas

def saldo_migratorio(ilha):
    return ilha.imigrantes - ilha.emigrantes

def resumo_migracao(ilha):
    return {
        "imigrantes": ilha.imigrantes,
        "emigrantes": ilha.emigrantes,
        "saldo": saldo_migratorio(ilha)
    }

Overwriting migracao.py


### **Recursos**

In [71]:
%%writefile recursos.py

# ==========================================
# RECURSOS V13 (Condições Iguais e Justas para Todos)
# ==========================================

def fator_recursos(ilha):
    energia_ratio = min(1.0, ilha.energia / 50000)
    minerio_ratio = min(1.0, ilha.minerio / 50000)
    fator = min(energia_ratio, minerio_ratio)
    return max(0.25, fator)


# ==========================================
# PRODUÇÃO AGRÍCOLA (Universal e Equilibrada)
# ==========================================

def produzir_alimentos(ilha):
    # Base de trabalho rural unificada para todas as ilhas sem distinção de regime
    trabalho_rural = ilha.empregados + (ilha.informais * 0.5) + (ilha.populacao * 0.08)

    producao = (
        trabalho_rural *
        ilha.produtividade *
        ilha.incentivo *
        1.8  # Fator multiplicador padrão aplicado igualmente a todas as ilhas
    )

    ilha.producao_agricola = producao
    return producao


# ==========================================
# PRODUÇÃO INDUSTRIAL
# ==========================================

def produzir_industria(ilha):
    fator = fator_recursos(ilha)

    producao = (
        ilha.empregados *
        ilha.produtividade *
        fator *
        0.35
    )

    ilha.producao_industrial = producao
    return producao


# ==========================================
# SERVIÇOS
# ==========================================

def produzir_servicos(ilha):
    producao = (
        ilha.empregados *
        ilha.produtividade *
        0.20
    )

    ilha.producao_servicos = producao
    return producao


# ==========================================
# CONSUMO DE ALIMENTOS
# ==========================================

def atualizar_alimentos(ilha):
    consumo = ilha.populacao * 0.85

    ilha.alimentos += ilha.producao_agricola
    ilha.alimentos -= consumo

    ilha.alimentos = max(
        0,
        ilha.alimentos
    )

    return ilha


# ==========================================
# CONSUMO DE ENERGIA E MINÉRIO
# ==========================================

def consumir_energia(ilha):
    consumo = ilha.producao_industrial * 0.08
    ilha.energia -= consumo
    ilha.energia = max(0, ilha.energia)
    return ilha

def consumir_minerio(ilha):
    consumo = ilha.producao_industrial * 0.05
    ilha.minerio -= consumo
    ilha.minerio = max(0, ilha.minerio)
    return ilha


# ==========================================
# REGENERAÇÃO
# ==========================================

def regenerar_recursos(ilha):
    ilha.energia += 500 + ilha.inovacao * 10
    ilha.minerio += 100 + ilha.inovacao * 2

    ilha.energia = min(200000, ilha.energia)
    ilha.minerio = min(200000, ilha.minerio)
    return ilha


# ==========================================
# CICLO COMPLETO
# ==========================================

def atualizar_recursos(ilha):
    produzir_alimentos(ilha)
    produzir_industria(ilha)
    produzir_servicos(ilha)
    atualizar_alimentos(ilha)
    consumir_energia(ilha)
    consumir_minerio(ilha)
    regenerar_recursos(ilha)
    return ilha

Overwriting recursos.py


### **Governo**

In [72]:
%%writefile governo.py

# ==========================================
# GOVERNO V10 (Fiscalmente Sustentável)
# ==========================================

def parametros_institucionais(nome_regime):
    configuracao = {
        "Socialismo": {
            "transparencia": 0.70,
            "fiscalizacao": 0.75,
            "concentracao_poder": 0.70
        },
        "Social Democracia": {
            "transparencia": 0.85,
            "fiscalizacao": 0.85,
            "concentracao_poder": 0.35
        },
        "Conservador": {
            "transparencia": 0.75,
            "fiscalizacao": 0.70,
            "concentracao_poder": 0.50
        },
        "Liberal": {
            "transparencia": 0.80,
            "fiscalizacao": 0.65,
            "concentracao_poder": 0.40
        }
    }
    return configuracao[nome_regime]


def calcular_corrupcao(transparencia, fiscalizacao, concentracao_poder):
    corrupcao = (
        concentracao_poder * 0.40 +
        (1 - transparencia) * 0.30 +
        (1 - fiscalizacao) * 0.30
    )
    return max(0.02, min(0.40, corrupcao))


def calcular_juros(divida, pib):
    if pib <= 0:
        return 10.0
    relacao = divida / max(1, pib)
    juros = 4 + (relacao * 1.5)
    return max(2, min(15, juros))


def arrecadar_impostos(pib, imposto, corrupcao):
    arrecadacao_bruta = pib * imposto
    perda_corrupcao = arrecadacao_bruta * corrupcao
    arrecadacao_liquida = arrecadacao_bruta - perda_corrupcao
    return {
        "bruta": arrecadacao_bruta,
        "perdida": perda_corrupcao,
        "liquida": arrecadacao_liquida
    }


def calcular_gastos(arrecadacao, gasto_social, investimento_publico, idosos):
    # Previdência dimensionada de forma sustentável vinculada aos idosos e à arrecadação
    previdencia = min(arrecadacao * 0.4, idosss_custo := idosos * 0.1)

    gasto_saude_educacao = arrecadacao * (gasto_social + investimento_publico)
    gasto_total = gasto_saude_educacao + previdencia

    return {
        "total": gasto_total,
        "previdencia": previdencia
    }


def atualizar_divida(divida, arrecadacao, gastos):
    saldo = arrecadacao - gastos
    if saldo < 0:
        divida += abs(saldo) * 0.2  # Crescimento de dívida mais suave
    else:
        divida = max(0, divida - saldo)
    return divida


def atualizar_governo(ilha):
    inst = parametros_institucionais(ilha.nome)
    ilha.transparencia = inst["transparencia"]
    ilha.fiscalizacao = inst["fiscalizacao"]
    ilha.concentracao_poder = inst["concentracao_poder"]

    ilha.corrupcao = calcular_corrupcao(
        ilha.transparencia,
        ilha.fiscalizacao,
        ilha.concentracao_poder
    )

    arrec = arrecadar_impostos(
        ilha.pib,
        ilha.imposto,
        ilha.corrupcao
    )

    ilha.arrecadacao = arrec["liquida"]
    ilha.perda_corrupcao = arrec["perdida"]

    gastos = calcular_gastos(
        arrec["liquida"],
        ilha.gasto_social,
        ilha.investimento,
        ilha.idosos
    )

    ilha.gasto_publico = gastos["total"]
    ilha.previdencia = gastos["previdencia"]

    ilha.divida = atualizar_divida(
        ilha.divida,
        ilha.arrecadacao,
        ilha.gasto_publico
    )

    ilha.juros = calcular_juros(
        ilha.divida,
        ilha.pib
    )

    return ilha

Overwriting governo.py


### **Demografia**

In [73]:
%%writefile demografia.py

# ==========================================
# DEMOGRAFIA V11 (Definitiva)
# ==========================================

from indicadores import calcular_expectativa_vida

def calcular_natalidade(adultos, educacao, saude):
    taxa = (
        0.024
        - educacao / 12000
        + saude / 50000
    )
    taxa = max(0.008, min(0.04, taxa))
    nascimentos = adultos * taxa
    return nascimentos

def calcular_mortalidade(criancas, adultos, idosos, saude, fome_grave):
    mortes_criancas = criancas * (
        0.0025
        + (fome_grave / 100) * 0.002
    )
    mortes_adultos = adultos * (
        0.005
        + (fome_grave / 100) * 0.003
    )
    fator_saude_idosos = max(0.005, 0.018 - (saude / 5000))
    mortes_idosos = idosos * fator_saude_idosos

    return int(mortes_criancas), int(mortes_adultos), int(mortes_idosos)

def forca_trabalho(ilha):
    return int(ilha.adultos * 0.85)

def atualizar_demografia(ilha):
    ilha.expectativa_vida = calcular_expectativa_vida(ilha.saude)

    nascimentos = int(calcular_natalidade(
        ilha.adultos,
        ilha.educacao,
        ilha.saude
    ))

    m_criancas, m_adultos, m_idosos = calcular_mortalidade(
        ilha.criancas,
        ilha.adultos,
        ilha.idosos,
        ilha.saude,
        ilha.fome_grave
    )

    total_mortes = m_criancas + m_adultos + m_idosos

    transicao_criancas = int(ilha.criancas * 0.05)
    transicao_adultos = int(ilha.adultos * 0.03)
    transicao_idosos = int(ilha.idosos * 0.02) # Garantido explicitamente

    ilha.criancas = max(0, ilha.criancas + nascimentos - m_criancas - transicao_criancas)
    ilha.adultos = max(0, ilha.adultos + transicao_criancas - m_adultos - transicao_adultos)
    ilha.idosos = max(0, ilha.idosos + transicao_adultos - m_idosos)

    ilha.total_nascimentos += nascimentos
    ilha.total_mortes += total_mortes

    return ilha

Overwriting demografia.py


### **Comércio**

In [74]:
%%writefile comercio.py

# ==========================================
# COMÉRCIO V13 (Comércio Bilateral Ativo)
# ==========================================

def atualizar_comercio(ilhas):
    for ilha in ilhas:
        ilha.exportacoes = 0
        ilha.importacoes = 0
        ilha.saldo_comercial = 0

    # 1. Comércio Ativo de Alimentos (Exportadores vs. Importadores em Déficit)
    for importador in ilhas:
        consumo_imp = importador.populacao * 0.85

        # Se a ilha está com déficit alimentar severo ou abaixo do consumo seguro
        if importador.alimentos < (consumo_imp + 2000):
            deficit = (consumo_imp + 2000) - importador.alimentos

            for exportador in ilhas:
                if exportador == importador:
                    continue

                consumo_exp = exportador.populacao * 0.85
                excedente_exp = exportador.alimentos - (consumo_exp + 4000)

                if excedente_exp > 500 and deficit > 0:
                    quantidade_troca = min(excedente_exp, deficit, 1000)

                    if quantidade_troca > 0:
                        exportador.alimentos -= quantidade_troca
                        importador.alimentos += quantidade_troca

                        exportador.exportacoes += quantidade_troca
                        importador.importacoes += 0  # Registrado no balanço global

                        importador.importacoes += quantidade_troca
                        exportador.exportacoes += 0

                        deficit -= quantidade_troca
                        excedente_exp -= quantidade_troca

    # 2. Comércio de Minério
    for importador in ilhas:
        if importador.minerio < 90000:
            deficit_min = 100000 - importador.minerio
            for exportador in ilhas:
                if exportador == importador:
                    continue
                if exportador.minerio > 110000:
                    excedente_min = exportador.minerio - 100000
                    troca_min = min(excedente_min, deficit_min, 500)

                    if troca_min > 0:
                        exportador.minerio -= troca_min
                        importador.minerio += troca_min

                        exportador.exportacoes += int(troca_min * 1.5)
                        importador.importacoes += int(troca_min * 1.5)

                        deficit_min -= troca_min

    for ilha in ilhas:
        ilha.saldo_comercial = ilha.exportacoes - ilha.importacoes

    return ilhas

Overwriting comercio.py


### **Main**

In [67]:
%%writefile main.py

# ==========================================
# SIMULADOR SOCIOECONÔMICO - MAIN V12 (Com Eventos e CLI)
# ==========================================

import sys
import select
import time
import argparse
from demografia import atualizar_demografia
from economia import atualizar_economia
from governo import atualizar_governo
from recursos import atualizar_recursos
from comercio import atualizar_comercio
from migracao import atualizar_migracao
from indicadores import calcular_indicadores
from eventos import aplicar_eventos_aleatorios
import pandas as pd
import matplotlib.pyplot as plt

# Configuração do Pandas para exibir todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

class Ilha:
    def __init__(self, nome, populacao=10000, pib=100, capital=10000, empresas=10,
                 educacao=50, saude=50, inovacao=30, liberdade=70, imposto=0.25,
                 gasto_social=0.30, investimento=0.20, incentivo=1.0):
        self.nome = nome
        self.populacao = populacao
        self.criancas = int(populacao * 0.25)
        self.adultos = int(populacao * 0.65)
        self.idosos = int(populacao * 0.10)
        self.pib = pib
        self.capital = capital
        self.empresas = empresas
        self.empregados = 0
        self.informais = 0
        self.desempregados = 0
        self.desemprego = 0.0
        self.salario_medio = 0.0
        self.produtividade = 1.0
        self.educacao = educacao
        self.saude = saude
        self.inovacao = inovacao
        self.liberdade = liberdade
        self.imposto = imposto
        self.gasto_social = gasto_social
        self.investimento = investimento
        self.incentivo = incentivo
        self.nascimentos_acumulados = 0
        self.mortes_acumuladas = 0
        self.imigrantes = 0
        self.emigrantes = 0
        self.exportacoes = 0
        self.importacoes = 0
        self.saldo_comercial = 0
        self.alimentos = 5000.0
        self.energia = 100000.0
        self.minerio = 100000.0
        self.producao_agricola = 0.0
        self.producao_industrial = 0.0
        self.producao_servicos = 0.0
        self.fome_grave = 0.0
        self.pessoas_em_fome = 0
        self.expectativa_vida = 70.0
        self.transparencia = 0.8
        self.fiscalizacao = 0.8
        self.concentracao_poder = 0.4
        self.corrupcao = 0.1
        self.perda_corrupcao = 0.0
        self.juros = 10.0
        self.divida = 0.0
        self.felicidade = 50.0
        self.total_nascimentos = 0
        self.total_mortes = 0

def rodar_ano(ilhas, ano_atual):
    # Aplica eventos aleatórios do novo módulo
    msg_evento = aplicar_eventos_aleatorios(ilhas)
    if msg_evento and ano_atual % 10 == 0:
        print(f"[Ano {ano_atual}] {msg_evento}")

    for ilha in ilhas:
        atualizar_demografia(ilha)
        atualizar_economia(ilha)
        atualizar_recursos(ilha)
        atualizar_governo(ilha)
        calcular_indicadores(ilha)

    atualizar_comercio(ilhas)
    atualizar_migracao(ilhas)

def input_com_countdown(prompt, timeout=30, default=100):
    start_time = time.time()
    print(f"{prompt} [Padrão: {default}]", flush=True)

    while True:
        elapsed = time.time() - start_time
        remaining = int(timeout - elapsed)

        if remaining <= 0:
            print(f"\n[Tempo esgotado! Usando o valor padrão: {default} anos]")
            return default

        sys.stdout.write(f"\rDigite os anos ({remaining}s restantes): ")
        sys.stdout.flush()

        rlist, _, _ = select.select([sys.stdin], [], [], 1.0)

        if rlist:
            entrada = sys.stdin.readline().strip()
            print()
            if entrada == "":
                return default
            try:
                return int(entrada)
            except ValueError:
                print(f"[Entrada inválida! Usando o padrão: {default} anos]")
                return default

def main():
    parser = argparse.ArgumentParser(description="Simulador Socioeconômico V12")
    parser.add_argument("-t", "--anos", type=int, default=None, help="Quantidade de anos de simulação")
    parser.add_argument("-p", "--pronto", action="store_true", help="Executa automaticamente sem prompt")

    args, unknown = parser.parse_known_args()

    if args.anos is not None or args.pronto:
        anos = args.anos if args.anos is not None else 100
        print(f"[Modo Automático] Executando simulação por {anos} anos...")
    else:
        prompt_texto = "Digite a quantidade de anos de simulação"
        anos = input_com_countdown(prompt_texto, timeout=30, default=100)

    ilhas = [
        Ilha("Socialismo", liberdade=30, imposto=0.45, gasto_social=0.55),
        Ilha("Social Democracia", liberdade=65, imposto=0.35, gasto_social=0.40),
        Ilha("Conservador", liberdade=55, imposto=0.20, gasto_social=0.20),
        Ilha("Liberal", liberdade=90, imposto=0.12, gasto_social=0.10)
    ]

    for ano in range(anos):
        rodar_ano(ilhas, ano + 1)

    dados = []
    for i in ilhas:
        dados.append({
            "Modelo": i.nome,
            "Populacao": int(i.populacao),
            "PIB": int(i.pib),
            "PIB_per_capita": round(i.pib / max(1, i.populacao), 2),
            "Capital": int(i.capital),
            "Empresas": i.empresas,
            "Empregados": i.empregados,
            "Desemprego_%": round(i.desemprego, 2),
            "Salario_Medio": round(i.salario_medio, 2),
            "Produtividade": round(i.produtividade, 2),
            "Nascimentos": i.total_nascimentos,
            "Mortes": i.total_mortes,
            "Imigrantes": i.imigrantes,
            "Emigrantes": i.emigrantes,
            "Saldo_Migratorio": i.imigrantes - i.emigrantes,
            "Exportacoes": int(i.exportacoes),
            "Importacoes": int(i.importacoes),
            "Saldo_Comercial": int(i.exportacoes - i.importacoes),
            "Alimentos": int(i.alimentos),
            "Energia": int(i.energia),
            "Minerio": int(i.minerio),
            "Prod_Agro": int(i.producao_agricola),
            "Prod_Industria": int(i.producao_industrial),
            "Prod_Servicos": int(i.producao_servicos),
            "Fome_Grave_%": round(i.fome_grave, 2),
            "Pessoas_em_Fome": i.pessoas_em_fome,
            "Expectativa_Vida": round(i.expectativa_vida, 1),
            "Corrupcao_%": round(i.corrupcao * 100, 1),
            "Transparencia": i.transparencia,
            "Fiscalizacao": i.fiscalizacao,
            "Perda_Corrupcao": int(i.perda_corrupcao),
            "Juros_%": round(i.juros, 2),
            "Divida": int(i.divida),
            "Felicidade": round(i.felicidade, 2)
        })

    df = pd.DataFrame(dados)
    print(f"\n=== RESULTADO FINAL V12 ({anos} ANOS) ===")
    print(df)

    # Gráfico comparativo de Felicidade
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(df["Modelo"], df["Felicidade"], color='skyblue', alpha=0.8)
    ax.set_ylabel("Felicidade Média")
    ax.set_title(f"Comparativo de Felicidade entre Modelos ({anos} Anos com Eventos)")
    plt.show()

if __name__ == "__main__":
    main()

Overwriting main.py


### **Indicadores**

In [45]:
%%writefile indicadores.py

# ==========================================
# INDICADORES V11
# ==========================================

def calcular_expectativa_vida(saude):
    ev = 50 + (saude / 100) * 25
    return max(50.0, min(85.0, ev))

def calcular_atratividade(felicidade, salario, desemprego, expectativa_vida):
    score = (
        felicidade * 0.4 +
        salario * 15 * 0.3 +
        (100 - desemprego) * 0.2 +
        expectativa_vida * 0.1
    )
    return max(0.0, score)

def calcular_felicidade(ilha):
    fator_fome = max(0, 100 - (ilha.fome_grave * 2))
    fator_desemprego = max(0, 100 - ilha.desemprego)
    fator_saude = (ilha.expectativa_vida / 85) * 100

    felicidade = (
        fator_fome * 0.35 +
        fator_desemprego * 0.25 +
        fator_saude * 0.25 +
        ilha.liberdade * 0.15
    )
    return max(0.0, min(100.0, felicidade))

def calcular_fome(ilha):
    # Fome proporcional à população que não consegue suprir alimentos
    deficit = max(0, (ilha.populacao * 0.85) - ilha.alimentos)
    if deficit > 0:
        percentual = (deficit / (ilha.populacao * 0.85)) * 100
        ilha.fome_grave = min(100.0, percentual)
        ilha.pessoas_em_fome = int(ilha.populacao * (ilha.fome_grave / 100))
    else:
        ilha.fome_grave = 0.0
        ilha.pessoas_em_fome = 0
    return ilha.fome_grave

def calcular_indicadores(ilha):
    calcular_fome(ilha)
    ilha.expectativa_vida = calcular_expectativa_vida(ilha.saude)
    ilha.felicidade = calcular_felicidade(ilha)
    return ilha

Overwriting indicadores.py


### **Economia**

In [51]:
%%writefile economia.py

# ==========================================
# ECONOMIA V10 (Com Setor Informal)
# ==========================================

from demografia import forca_trabalho


# ==========================================\n
# PRODUTIVIDADE
# ==========================================\n

def calcular_produtividade(
    educacao,
    inovacao,
    capital
):
    produtividade = (
        1 +
        educacao / 120 +
        inovacao / 150 +
        capital / 100000
    )
    return max(0.5, produtividade)


# ==========================================\n
# CONFIANÇA ECONÔMICA
# ==========================================\n

def confianca_economica(ilha):
    confianca = (
        ilha.liberdade * 0.40 +
        (100 - ilha.desemprego) / 100 * 0.30 +
        ilha.educacao / 100 * 0.15 +
        ilha.inovacao / 100 * 0.15
    )
    return max(0.10, min(1.50, confianca))


# ==========================================\n
# EMPRESAS
# ==========================================\n

def atualizar_empresas(ilha):
    confianca = confianca_economica(ilha)

    novas = int(
        ilha.liberdade *
        confianca *
        ilha.inovacao *
        0.05
    )

    falencias = int(
        ilha.juros / 4 +
        ilha.desemprego / 20
    )

    ilha.empresas += (novas - falencias)
    ilha.empresas = max(10, ilha.empresas)

    return ilha


# ==========================================\n
# EMPREGO FORMAL E INFORMAL (ESTILO BRASIL)
# ==========================================\n

def gerar_empregos(ilha):
    trabalhadores = forca_trabalho(ilha)

    # Vagas formais geradas pela capacidade econômica
    vagas_formais = int(
        (ilha.capital * ilha.liberdade) / 60
    )

    ilha.empregados = min(trabalhadores, vagas_formais)

    restante_sem_vaga = trabalhadores - ilha.empregados

    # Uma parte dos sem-vaga vai para a informalidade (bicos / conta própria)
    # No Brasil, a informalidade absorve grande parte do excedente
    ilha.informais = int(restante_sem_vaga * 0.70)

    # O restante real fica totalmente desempregado
    ilha.desempregados = max(0, restante_sem_vaga - ilha.informais)

    # Taxa de desemprego oficial considera apenas quem não tem absolutamente nada nem informal
    ilha.desemprego = (
        ilha.desempregados / max(1, trabalhadores)
    ) * 100

    return ilha


# ==========================================\n
# PRODUÇÃO
# ==========================================\n

def atualizar_produtividade(ilha):
    ilha.produtividade = calcular_produtividade(
        ilha.educacao,
        ilha.inovacao,
        ilha.capital
    )
    return ilha


# ==========================================\n
# SALÁRIOS E LUCROS (Considerando rendas formais e informais)
# ==========================================\n

def distribuir_renda(ilha):
    distribuicao = {
        "Socialismo": 0.85,
        "Social Democracia": 0.75,
        "Conservador": 0.65,
        "Liberal": 0.55
    }

    percentual_salario = distribuicao[ilha.nome]

    # Massa salarial formal + estimativa menor gerada pelos informais
    massa_formal = ilha.pib * percentual_salario
    massa_informal = ilha.informais * 0.4  # Informais ganham menos

    massa_salarial_total = massa_formal + massa_informal
    lucro_total = max(0, ilha.pib - massa_salarial_total)

    ilha.massa_salarial = massa_salarial_total
    ilha.lucro_total = lucro_total

    total_trabalhadores_com_renda = max(1, ilha.empregados + ilha.informais)
    ilha.salario_medio = massa_salarial_total / total_trabalhadores_com_renda

    return ilha


# ==========================================\n
# CAPITAL
# ==========================================\n

def acumular_capital(ilha):
    reinvestimento = (
        ilha.lucro_total *
        0.25 *
        (1 - ilha.juros / 100)
    )
    reinvestimento = max(0, reinvestimento)
    ilha.capital += reinvestimento
    return ilha


# ==========================================\n
# PIB PER CAPITA
# ==========================================\n

def pib_per_capita(ilha):
    return ilha.pib / max(1, ilha.populacao)


# ==========================================\n
# ATUALIZAÇÃO ECONÔMICA
# ==========================================\n

def atualizar_economia(ilha):
    atualizar_empresas(ilha)
    gerar_empregos(ilha)
    atualizar_produtividade(ilha)

    # PIB impulsionado por formais e parcialmente pelos informais
    trabalho_efetivo = ilha.empregados + (ilha.informais * 0.5)
    ilha.pib = trabalho_efetivo * ilha.produtividade

    distribuir_renda(ilha)
    acumular_capital(ilha)

    return ilha

Overwriting economia.py


### Eventos

In [65]:
%%writefile eventos.py

# ==========================================
# EVENTOS E CRISES V11
# ==========================================

import random

def aplicar_eventos_aleatorios(ilhas):
    # Sorteia a chance de ocorrer um evento crítico a cada ano (ex: 25% de chance)
    if random.random() > 0.25:
        return None

    evento_tipo = random.choice([
        "seca_severa",
        "boom_tecnologico",
        "crise_energetica",
        "boom_agricola"
    ])

    if evento_tipo == "seca_severa":
        # Afeta a produção agrícola de todas as ilhas severamente
        for ilha in ilhas:
            ilha.alimentos = max(0, ilha.alimentos * 0.75)
        return "⚠️ ALERTA: Uma Seca Severa atingiu a região, reduzindo os estoques de alimentos em 25%!"

    elif evento_tipo == "boom_tecnologico":
        # Impulsiona a inovação e produtividade temporariamente
        ilha_favorecida = random.choice(ilhas)
        ilha_favorecida.inovacao += 15
        ilha_favorecida.produtividade *= 1.10
        return f"🚀 BOOM TECNOLÓGICO: A ilha de {ilha_favorecida.nome} registrou avanços tecnológicos expressivos!"

    elif evento_tipo == "crise_energetica":
        # Reduz as reservas de energia de todas as ilhas
        for ilha in ilhas:
            ilha.energia = max(0, ilha.energia * 0.80)
        return "⚡ CRISE ENERGÉTICA: Falhas na rede afetaram as reservas de energia do continente (-20%)!"

    elif evento_tipo == "boom_agricola":
        # Aumenta a safra de alimentos
        for ilha in ilhas:
            ilha.alimentos += 5000
        return "🌾 BOOM AGRÍCOLA: Condições climáticas ideais geraram excedentes extras de safra para todas as ilhas!"

    return None

Writing eventos.py


# **Executar**

In [76]:
!python main.py -p

[Modo Automático] Executando simulação por 100 anos...
[Ano 20] 🌾 BOOM AGRÍCOLA: Condições climáticas ideais geraram excedentes extras de safra para todas as ilhas!
[Ano 30] ⚡ CRISE ENERGÉTICA: Falhas na rede afetaram as reservas de energia do continente (-20%)!
[Ano 50] 🚀 BOOM TECNOLÓGICO: A ilha de Conservador registrou avanços tecnológicos expressivos!
[Ano 70] 🚀 BOOM TECNOLÓGICO: A ilha de Socialismo registrou avanços tecnológicos expressivos!
[Ano 100] 🌾 BOOM AGRÍCOLA: Condições climáticas ideais geraram excedentes extras de safra para todas as ilhas!

=== RESULTADO FINAL V12 (100 ANOS) ===
              Modelo  Populacao   PIB  PIB_per_capita  Capital  Empresas  Empregados  Desemprego_%  Salario_Medio  Produtividade  Nascimentos  Mortes  Imigrantes  Emigrantes  Saldo_Migratorio  Exportacoes  Importacoes  Saldo_Comercial  Alimentos  Energia  Minerio  Prod_Agro  Prod_Industria  Prod_Servicos  Fome_Grave_%  Pessoas_em_Fome  Expectativa_Vida  Corrupcao_%  Transparencia  Fiscalizacao 